This notebook builds the weekly Theater District, control zone, and HVFHV datasets and joins them against Broadway weekly grosses. All statistical analysis, significance testing, and visualisation happens in `analysis.ipynb`.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd

spark = (SparkSession.builder
    .appName("Broadway_Weekly_Grosses")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/23 19:56:52 WARN Utils: Your hostname, LAPTOP-V00NHVKA, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/23 19:56:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/23 19:56:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/23 19:56:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/08/23 19:56:54 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Read the weekly grosses file.

In [2]:
weekly_grosses = spark.read.csv(
    "../data/broadway_data/Weekly_grosses.csv",
    header=True,
    inferSchema=False
)

weekly_grosses.limit(5)

26/08/23 19:56:58 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: ID, Week Ending, Gross, Attendance, % Capacity, 
 Schema: ID, Week Ending, Gross, Attendance, % Capacity, _c5
Expected: _c5 but found: 
CSV file: file:///mnt/c/Users/caiji/Downloads/ADS_Project_1/data/broadway_data/Weekly_grosses.csv
26/08/23 19:56:59 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: ID, Week Ending, Gross, Attendance, % Capacity, 
 Schema: ID, Week Ending, Gross, Attendance, % Capacity, _c5
Expected: _c5 but found: 
CSV file: file:///mnt/c/Users/caiji/Downloads/ADS_Project_1/data/broadway_data/Weekly_grosses.csv


ID,Week Ending,Gross,Attendance,% Capacity,_c5
1,"Feb 9, 2025","$28,135,565","247,887",94%,NULL
2,"Feb 2, 2025","$31,199,599","259,126",94%,NULL
3,"Jan 26, 2025","$29,614,804","246,230",91%,NULL
4,"Jan 19, 2025","$33,372,166","250,806",91%,NULL
5,"Jan 12, 2025","$32,176,529","254,320",92%,NULL


Check schema.

In [3]:
weekly_grosses.printSchema()

root
 |-- ID: string (nullable = true)
 |-- Week Ending: string (nullable = true)
 |-- Gross: string (nullable = true)
 |-- Attendance: string (nullable = true)
 |-- % Capacity: string (nullable = true)
 |-- _c5: string (nullable = true)



Change datatypes to correct ones and clean unwanted symbols.

In [4]:
weekly_clean = (
    weekly_grosses
    .withColumn(
        "Week Ending",
        F.to_date(F.col("Week Ending"), "MMM d, yyyy")
    )
    .withColumn(
        "Gross",
        F.regexp_replace(F.col("Gross"), "[$,]", "").cast("double")
    )
    .withColumn(
        "Attendance",
        F.regexp_replace(F.col("Attendance"), ",", "").cast("double")
    )
    .withColumn(
        "% Capacity",
        F.regexp_replace(F.col("% Capacity"), "%", "").cast("double") / 100
    )
)
weekly_clean.printSchema()

root
 |-- ID: string (nullable = true)
 |-- Week Ending: date (nullable = true)
 |-- Gross: double (nullable = true)
 |-- Attendance: double (nullable = true)
 |-- % Capacity: double (nullable = true)
 |-- _c5: string (nullable = true)



Check that filter did not turn any relevant data into NULL.

In [5]:
weekly_clean.filter(
    (F.year("Week Ending") == 2024) &
    F.col("Week Ending").isNull()
).count()

0

Filter to only include 2024 data.

In [6]:
weekly_grosses_2024 = weekly_clean.filter(
    F.year("Week Ending") == 2024
)

weekly_grosses_2024.count()

52

Check for any gaps by checking if the difference between each instance is 7 days.

In [7]:
from pyspark.sql.window import Window

window = Window.orderBy("Week Ending")

weekly_check = weekly_grosses_2024.withColumn(
    "previous_week",
    F.lag("Week Ending").over(window)
).withColumn(
    "days_since_previous",
    F.datediff("Week Ending", "previous_week")
)

weekly_check.filter(
    F.col("days_since_previous") != 7
).show()

26/08/23 19:57:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/23 19:57:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/23 19:57:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/23 19:57:00 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: ID, Week Ending, Gross, Attendance, % Capacity, 
 Schema: ID, Week Ending, Gross, Attendance, % Capacity, _c5
Expected: _c5 but found: 
CSV file: file:///mnt/c/Users/caiji/Downloads/ADS_Project_1/data/broadway_data/Weekly_grosses.csv
26/08/23 19:57:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/23 19:5

+---+-----------+-----+----------+----------+---+-------------+-------------------+
| ID|Week Ending|Gross|Attendance|% Capacity|_c5|previous_week|days_since_previous|
+---+-----------+-----+----------+----------+---+-------------+-------------------+
+---+-----------+-----+----------+----------+---+-------------+-------------------+



Rename columns in preparation to join with tlc data.

In [8]:
weekly_grosses_2024 = (
    weekly_grosses_2024
    .withColumnRenamed("Week Ending", "week_ending")
    .withColumnRenamed("Attendance", "attendance")
    .withColumnRenamed("Gross", "gross")
)

In [9]:
trips = spark.read.parquet(
    "../data/curated/yellow_taxi"
)
trips.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [10]:
trips_with_week = trips.withColumn(
    "week_ending",
    F.to_date(
        F.date_add(
            F.to_date("tpep_pickup_datetime"),
            6 - ((F.dayofweek("tpep_pickup_datetime") + 5) % 7) # formula ensures that the week ends on that week's Sunday
        )
    )
)
trips_with_week  = trips_with_week.filter(
    (F.col("week_ending") >= "2024-01-01") &
    (F.col("week_ending") <= "2024-12-31")
)
trips_with_week.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- week_ending: date (nullable = true)



In [11]:
trips_with_week.select(
    "tpep_pickup_datetime",
    "week_ending"
).orderBy(
    "tpep_pickup_datetime"
).limit(5)

tpep_pickup_datetime,week_ending
2024-01-01 00:01:25,2024-01-07
2024-01-01 00:01:36,2024-01-07
2024-01-01 00:01:58,2024-01-07
2024-01-01 00:02:16,2024-01-07
2024-01-01 00:02:16,2024-01-07


In [12]:
weekly_trips = trips_with_week.groupBy(
    "week_ending",
    "PULocationID"
).agg(
    F.count("*").alias("trip_count"),
    F.sum("fare_amount").alias("total_fare"),
    F.sum("tip_amount").alias("total_tip")
)
weekly_trips.printSchema()

root
 |-- week_ending: date (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- trip_count: long (nullable = false)
 |-- total_fare: double (nullable = true)
 |-- total_tip: double (nullable = true)



In [13]:
theaters_zones = pd.read_csv(
    "../data/broadway_data/theaters_zones.csv"
)

theater_zone_ids = (
    theaters_zones["LocationID"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

theater_zone_ids

[230, 48, 100, 142]

In [14]:
theater_weekly = (
    weekly_trips
    .filter(F.col("PULocationID").isin(theater_zone_ids))
    .filter(
        (F.col("week_ending") >= "2024-01-07") &
        (F.col("week_ending") <= "2024-12-29")
    )
    .groupBy("week_ending")
    .agg(
        F.sum("trip_count").alias("weekly_trips"),
        F.sum("total_fare").alias("weekly_fares"),
        F.sum("total_tip").alias("weekly_tips")
    )
    .orderBy("week_ending")
)

In [15]:
theater_weekly.count()

52

In [16]:
theater_weekly.show(5)

+-----------+------------+------------------+------------------+
|week_ending|weekly_trips|      weekly_fares|       weekly_tips|
+-----------+------------+------------------+------------------+
| 2024-01-07|       74576|1195884.9300000053| 212435.7300000011|
| 2024-01-14|       77114| 1154248.070000001|225651.86000000255|
| 2024-01-21|       74916|1147713.3299999977|224299.45000000193|
| 2024-01-28|       77687|1182120.8999999985|230067.62000000203|
| 2024-02-04|       75213|1156082.9499999965|227796.55000000107|
+-----------+------------+------------------+------------------+
only showing top 5 rows


In [17]:
weekly_joined = theater_weekly.join(
    weekly_grosses_2024,
    on="week_ending",
    how="inner"
)

In [18]:
weekly_joined.count()

52

In [19]:
weekly_joined.orderBy("week_ending").show(10, truncate=False)

26/08/23 19:57:09 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: ID, Week Ending, Gross, Attendance, % Capacity, 
 Schema: ID, Week Ending, Gross, Attendance, % Capacity, _c5
Expected: _c5 but found: 
CSV file: file:///mnt/c/Users/caiji/Downloads/ADS_Project_1/data/broadway_data/Weekly_grosses.csv


+-----------+------------+------------------+------------------+---+-----------+----------+----------+----+
|week_ending|weekly_trips|weekly_fares      |weekly_tips       |ID |gross      |attendance|% Capacity|_c5 |
+-----------+------------+------------------+------------------+---+-----------+----------+----------+----+
|2024-01-07 |74576       |1195884.9300000053|212435.7300000011 |58 |2.9681396E7|219787.0  |0.93      |NULL|
|2024-01-14 |77114       |1154248.070000001 |225651.86000000255|57 |2.7657991E7|218287.0  |0.9       |NULL|
|2024-01-21 |74916       |1147713.3299999977|224299.45000000193|56 |2.3589174E7|204612.0  |0.86      |NULL|
|2024-01-28 |77687       |1182120.8999999985|230067.62000000203|55 |2.4615726E7|217847.0  |0.92      |NULL|
|2024-02-04 |75213       |1156082.9499999965|227796.55000000107|54 |2.3493675E7|213281.0  |0.93      |NULL|
|2024-02-11 |76377       |1199768.8899999938|232740.41000000207|53 |2.1192251E7|187573.0  |0.86      |NULL|
|2024-02-18 |78013       |12

In [20]:
plot_data = weekly_joined.orderBy("week_ending").toPandas()

26/08/23 19:57:10 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: ID, Week Ending, Gross, Attendance, % Capacity, 
 Schema: ID, Week Ending, Gross, Attendance, % Capacity, _c5
Expected: _c5 but found: 
CSV file: file:///mnt/c/Users/caiji/Downloads/ADS_Project_1/data/broadway_data/Weekly_grosses.csv


In [21]:
plot_data.head()

,week_ending,weekly_trips,weekly_fares,weekly_tips,ID,gross,attendance,% Capacity,_c5
0,2024-01-07,74576,1195884.93,212435.73,58,29681396.0,219787.0,0.93,None
1,2024-01-14,77114,1154248.07,225651.86,57,27657991.0,218287.0,0.90,None
2,2024-01-21,74916,1147713.33,224299.45,56,23589174.0,204612.0,0.86,None
3,2024-01-28,77687,1182120.90,230067.62,55,24615726.0,217847.0,0.92,None
4,2024-02-04,75213,1156082.95,227796.55,54,23493675.0,213281.0,0.93,None


## Control zone (Murray Hill) — build the comparable non-theater weekly dataset

Same process as the Theater District above, applied to a comparable Manhattan zone with no Broadway theaters, so the analysis notebook can check whether any relationship with attendance is theater-specific or reflects general Midtown/holiday seasonality.

In [22]:
control_trips = spark.read.parquet("../data/curated/control_zone")

control_zone_name = "Murray Hill"

control_with_week = control_trips.withColumn(
    "week_ending",
    F.to_date(
        F.date_add(
            F.to_date("tpep_pickup_datetime"),
            6 - ((F.dayofweek("tpep_pickup_datetime") + 5) % 7)
        )
    )
).filter(
    (F.col("week_ending") >= "2024-01-07") &
    (F.col("week_ending") <= "2024-12-29")
)

control_weekly = (
    control_with_week
    .groupBy("week_ending")
    .agg(F.count("*").alias("weekly_trips"))
    .orderBy("week_ending")
)

control_joined = control_weekly.join(
    weekly_grosses_2024, on="week_ending", how="inner"
)

control_plot_data = control_joined.orderBy("week_ending").toPandas()
control_plot_data.head()

26/08/23 19:57:12 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: ID, Week Ending, Gross, Attendance, % Capacity, 
 Schema: ID, Week Ending, Gross, Attendance, % Capacity, _c5
Expected: _c5 but found: 
CSV file: file:///mnt/c/Users/caiji/Downloads/ADS_Project_1/data/broadway_data/Weekly_grosses.csv


,week_ending,weekly_trips,ID,gross,attendance,% Capacity,_c5
0,2024-01-07,16017,58,29681396.0,219787.0,0.93,None
1,2024-01-14,18769,57,27657991.0,218287.0,0.90,None
2,2024-01-21,18925,56,23589174.0,204612.0,0.86,None
3,2024-01-28,20361,55,24615726.0,217847.0,0.92,None
4,2024-02-04,19943,54,23493675.0,213281.0,0.93,None


## HVFHV (Uber/Lyft/Via/Juno) — build the Theater District weekly dataset

In [23]:
fhvhv_trips = spark.read.parquet("../data/curated/fhvhv")
fhvhv_trips.printSchema()


root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_

In [24]:
fhvhv_with_week = fhvhv_trips.withColumn(
    "week_ending",
    F.to_date(
        F.date_add(
            F.to_date("pickup_datetime"),
            6 - ((F.dayofweek("pickup_datetime") + 5) % 7)
        )
    )
).filter(
    (F.col("week_ending") >= "2024-01-07") &
    (F.col("week_ending") <= "2024-12-29")
)

fhvhv_weekly = (
    fhvhv_with_week
    .filter(F.col("PULocationID").isin(theater_zone_ids))
    .groupBy("week_ending")
    .agg(
        F.count("*").alias("weekly_trips"),
        F.sum("base_passenger_fare").alias("weekly_fares"),
        F.sum("tips").alias("weekly_tips")
    )
    .orderBy("week_ending")
)

fhvhv_joined = fhvhv_weekly.join(
    weekly_grosses_2024, on="week_ending", how="inner"
)

fhvhv_plot_data = fhvhv_joined.orderBy("week_ending").toPandas()

fhvhv_plot_data.head()

26/08/23 19:57:18 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: ID, Week Ending, Gross, Attendance, % Capacity, 
 Schema: ID, Week Ending, Gross, Attendance, % Capacity, _c5
Expected: _c5 but found: 
CSV file: file:///mnt/c/Users/caiji/Downloads/ADS_Project_1/data/broadway_data/Weekly_grosses.csv


,week_ending,weekly_trips,weekly_fares,weekly_tips,ID,gross,attendance,% Capacity,_c5
0,2024-01-07,142821,4387309.33,260832.87,58,29681396.0,219787.0,0.93,None
1,2024-01-14,159656,4801698.33,310212.41,57,27657991.0,218287.0,0.90,None
2,2024-01-21,173966,5318238.30,347742.61,56,23589174.0,204612.0,0.86,None
3,2024-01-28,164684,5073782.55,320388.71,55,24615726.0,217847.0,0.92,None
4,2024-02-04,159106,4947887.02,315346.92,54,23493675.0,213281.0,0.93,None


## Save the three preprocessed weekly datasets for the analysis notebook

Saving as plain CSV means `analysis.ipynb` only needs pandas/scipy/matplotlib — no Spark session required — which keeps the analysis notebook fast to open and fully decoupled from the preprocessing pipeline.

In [25]:
import os

analysis_ready_dir = "../data/analysis_ready"
os.makedirs(analysis_ready_dir, exist_ok=True)

plot_data.to_csv(f"{analysis_ready_dir}/theater_district_weekly.csv", index=False)
control_plot_data.to_csv(f"{analysis_ready_dir}/control_zone_weekly.csv", index=False)
fhvhv_plot_data.to_csv(f"{analysis_ready_dir}/fhvhv_theater_district_weekly.csv", index=False)

print("Saved:")
print(f"  {analysis_ready_dir}/theater_district_weekly.csv       ({len(plot_data)} rows)")
print(f"  {analysis_ready_dir}/control_zone_weekly.csv           ({len(control_plot_data)} rows)")
print(f"  {analysis_ready_dir}/fhvhv_theater_district_weekly.csv ({len(fhvhv_plot_data)} rows)")

Saved:
  ../data/analysis_ready/theater_district_weekly.csv       (52 rows)
  ../data/analysis_ready/control_zone_weekly.csv           (52 rows)
  ../data/analysis_ready/fhvhv_theater_district_weekly.csv (52 rows)
